# Review SpecMine specs by hand

Pick what to review in the setup cell, in one of three ways:

- `REVIEW_MODE = "informed"` or `"blind"` walks through `informed_batch.csv` or `blind_batch.csv` in `data/labeled_data/`, using the rows in `REVIEW_BATCH = (start, count)`. This is the mode for hand-labeling.
- With `REVIEW_MODE = None`, `REVIEW_BATCH` walks through rows of `LABELS_CSV` instead.
- `SPEC_IDS` shows specific specs.
- `REPO_IDS` shows every spec of the listed repositories.

For each spec the notebook shows:

1. **Repository and file card**: repo, stars, language, spec tool and role, dates, and the first and last commit to the file.
2. **Provenance evidence**: AI involvement recorded in commit metadata (co-author trailers, agent commit identities, agent-driven tool paths). This never reads the spec text.
3. **Features to check**: surface features from the labeling rubric (categories A to I, plus human indicators) with counts and verbatim examples.
4. **The Markdown**, escaped and line-numbered, with those features highlighted in place.

The highlights are regex matches meant to draw your eye, not verdicts. Read the hit in context before counting it.

**Blind labeling:** `REVIEW_MODE = "blind"` shows only the spec text and the rubric highlights, which is the same information the LLM labelers got. The repository name, file path, spec tool, commits, provenance evidence and stored LLM label are all hidden, because each of them can give the answer away. Don't open the spec on GitHub while labeling blind.

Record labels in `data/labeled_data/human_labels.csv`. Cached metadata, features and Markdown are reused from `data`; only a missing required file is downloaded.

In [ ]:
import html
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import (
    download_spec_files,
    load_data,
    load_feature_data,
)
from specmine.review import (
    TELL_CATEGORIES,
    provenance_evidence,
    render_feature_strip,
    render_legend,
    render_markdown,
    render_metadata_card,
    text_tells,
)

DATA_DIR = project_root / "data"
LABELS_CSV = project_root / "data/labeled_data/real_labels.csv"
BATCH_FILES = {
    "informed": project_root / "data/labeled_data/informed_batch.csv",
    "blind": project_root / "data/labeled_data/blind_batch.csv",
}

# Which labeling pass you are doing: "informed", "blind", or None.
# "blind" hides everything except the spec text: repo name, path, tool,
# commits, provenance and the stored LLM label.
REVIEW_MODE = "blind"
# Rows to show, as (start, count). Rows of the batch file for REVIEW_MODE,
# or of LABELS_CSV when REVIEW_MODE is None. None shows the whole batch file.
REVIEW_BATCH = (73, 10)
# Used only when REVIEW_MODE and REVIEW_BATCH are both None.
REPO_IDS = []  # Example: [123456789, 987654321] or "123456789, 987654321"
SPEC_IDS = []  # Example: ["4063ec9b808116cb"]

# What to show for each spec. Blind mode turns the first two off.
SHOW_PROVENANCE = True  # Repo, path, commit messages and AI-involvement evidence.
SHOW_LLM_LABEL = True  # The label and reasoning stored in LABELS_CSV.
HIGHLIGHT_TELLS = True  # Highlight rubric features in the Markdown.


In [ ]:
def normalize_inputs(values):
    if values is None:
        return []
    if isinstance(values, str):
        values = values.split(",")
    elif not isinstance(values, (list, tuple, set)):
        values = [values]
    normalized = [str(value).strip() for value in values if str(value).strip()]
    return list(dict.fromkeys(normalized))


review_order = None
review_csv = LABELS_CSV
if REVIEW_MODE is not None:
    if REVIEW_MODE not in BATCH_FILES:
        raise ValueError(f"REVIEW_MODE must be one of {sorted(BATCH_FILES)} or None.")
    review_csv = BATCH_FILES[REVIEW_MODE]
    if REVIEW_BATCH is None:
        REVIEW_BATCH = (0, len(pd.read_csv(review_csv, dtype=str)))
BLIND = REVIEW_MODE == "blind"
if BLIND:
    SHOW_PROVENANCE = False
    SHOW_LLM_LABEL = False

if REVIEW_BATCH is not None:
    batch_start, batch_count = REVIEW_BATCH
    batch = pd.read_csv(review_csv, dtype=str).iloc[
        batch_start : batch_start + batch_count
    ]
    if batch.empty:
        raise ValueError(f"REVIEW_BATCH {REVIEW_BATCH} selects no rows of {review_csv}.")
    requested_repo_ids = []
    requested_spec_ids = normalize_inputs(batch["spec_id"].tolist())
    review_order = {spec_id: index for index, spec_id in enumerate(requested_spec_ids)}
    print(
        f"Reviewing rows {batch_start}-{batch_start + len(batch) - 1} of {review_csv.name}"
        + (" (blind)." if BLIND else ".")
    )
else:
    requested_repo_ids = normalize_inputs(REPO_IDS)
    requested_spec_ids = normalize_inputs(SPEC_IDS)
    if bool(requested_repo_ids) == bool(requested_spec_ids):
        raise ValueError(
            "Set REVIEW_BATCH, or one or more REPO_IDS or SPEC_IDS (not both)."
        )

metadata = load_data(data_dir=DATA_DIR)
if metadata is None or metadata.empty:
    raise RuntimeError("SpecMine metadata is empty or unavailable.")

required_columns = {"repo_id", "repo_name", "file_url_sha16", "file_path", "file_url"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise KeyError(f"SpecMine metadata is missing columns: {sorted(missing_columns)}")

if requested_spec_ids:
    selected_specs = metadata.loc[
        metadata["file_url_sha16"].astype("string").isin(requested_spec_ids)
    ].copy()
    found_spec_ids = set(selected_specs["file_url_sha16"].astype("string"))
    missing_spec_ids = sorted(set(requested_spec_ids) - found_spec_ids)
    if missing_spec_ids:
        raise ValueError(f"No SpecMine specs found for spec_id(s): {missing_spec_ids}")
else:
    requested_repo_values = pd.to_numeric(
        pd.Series(requested_repo_ids), errors="coerce"
    )
    if (
        requested_repo_values.isna().any()
        or not requested_repo_values.mod(1).eq(0).all()
    ):
        raise ValueError("Every REPO_ID must be an integer repository ID.")
    normalized_repo_ids = {str(int(value)) for value in requested_repo_values}
    repository_ids = (
        pd.to_numeric(metadata["repo_id"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )
    selected_specs = metadata.loc[repository_ids.isin(normalized_repo_ids)].copy()
    found_repo_ids = set(
        selected_specs["repo_id"].dropna().map(lambda value: str(int(value)))
    )
    missing_repo_ids = sorted(normalized_repo_ids - found_repo_ids)
    if missing_repo_ids:
        raise ValueError(f"No SpecMine specs found for repo_id(s): {missing_repo_ids}")

if selected_specs["file_url_sha16"].isna().any():
    raise ValueError("Selected metadata contains a spec with no spec_id.")
selected_specs = selected_specs.drop_duplicates(subset=["file_url_sha16"])
if not BLIND:
    print(
        f"Selected {len(selected_specs)} distinct spec(s) from {selected_specs['repo_id'].nunique()} repository/repositories."
    )

def review_sorted(frame):
    """Batch order when reviewing a batch, otherwise repo then path."""
    if review_order is None:
        return frame.sort_values(["repo_name", "file_path", "file_url_sha16"])
    return frame.sort_values(
        "file_url_sha16", key=lambda ids: ids.astype("string").map(review_order)
    )


overview_columns = (
    ["file_url_sha16"] if BLIND else ["repo_id", "repo_name", "file_url_sha16", "file_path"]
)
display(review_sorted(selected_specs)[overview_columns].reset_index(drop=True))

In [ ]:
features = load_feature_data(
    selected_specs[["file_url_sha16"]],
    data_dir=DATA_DIR,
)
feature_ids = features["file_url_sha16"].astype("string")
selected_ids = selected_specs["file_url_sha16"].astype("string")
missing_feature_ids = sorted(set(selected_ids) - set(feature_ids))
if missing_feature_ids:
    raise ValueError(
        f"No precomputed feature row found for spec ID(s): {missing_feature_ids[:10]}"
    )
features = features.loc[feature_ids.isin(selected_ids)].copy()
if features["file_url_sha16"].astype("string").duplicated().any():
    raise ValueError("Feature table contains duplicate rows for a selected spec ID.")
if set(features["file_url_sha16"].astype("string")) != set(selected_ids):
    raise ValueError("Selected specs and feature rows do not match one-to-one.")

downloaded_specs = download_spec_files(selected_specs, data_dir=DATA_DIR)
downloaded_ids = set(downloaded_specs["file_url_sha16"].astype("string"))
if downloaded_ids != set(selected_ids):
    unavailable_ids = sorted(set(selected_ids) - downloaded_ids)
    raise RuntimeError(
        f"Could not download selected spec ID(s): {unavailable_ids[:10]}"
    )

spec_details = selected_specs.merge(
    features,
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
).merge(
    downloaded_specs[["file_url_sha16", "local_path"]],
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
)
if len(spec_details) != len(selected_specs):
    raise RuntimeError("Not every selected spec has exactly one feature row and file.")
print("Feature rows matched one-to-one by spec_id.")

In [ ]:
llm_labels = {}
if SHOW_LLM_LABEL and LABELS_CSV.is_file():
    llm_labels = (
        pd.read_csv(LABELS_CSV, dtype=str)
        .fillna("")
        .drop_duplicates(subset=["spec_id"])
        .set_index("spec_id")
        .to_dict("index")
    )

ordered_details = review_sorted(spec_details).reset_index(drop=True)
for position, row in enumerate(ordered_details.to_dict("records"), start=1):
    spec_id = str(row["file_url_sha16"])
    spec_path = Path(row["local_path"])
    if not spec_path.is_file():
        raise FileNotFoundError(f"Downloaded spec file is missing: {spec_path}")
    markdown = spec_path.read_text(encoding="utf-8", errors="replace")

    source = "" if BLIND else f"{html.escape(str(row['repo_name']))} &middot; "
    display(
        HTML(
            f"<h3>{position} of {len(ordered_details)}: "
            f"{source}spec_id {html.escape(spec_id)}</h3>"
        )
    )
    if not BLIND:
        evidence = provenance_evidence(row) if SHOW_PROVENANCE else None
        display(HTML(render_metadata_card(row, evidence, show_commits=SHOW_PROVENANCE)))
        # content_family names the spec tool, so the strip is hidden when blind.
        display(HTML(render_feature_strip(row)))

    if spec_id in llm_labels:
        stored = llm_labels[spec_id]
        display(
            HTML(
                f"<p><b>Stored label:</b> {html.escape(stored.get('label', ''))} "
                f"(confidence {html.escape(stored.get('confidence', ''))})</p>"
            )
        )
        print(stored.get("reasoning", ""))

    tells = text_tells(markdown)
    display(HTML("<h4>Features to check</h4>"))
    if tells.empty:
        print("No rubric features matched. Read the text for voice and specifics.")
    else:
        with pd.option_context("display.max_colwidth", None):
            display(tells)

    display(HTML("<h4>Spec Markdown</h4>"))
    if HIGHLIGHT_TELLS and not tells.empty:
        display(HTML(render_legend(tells["category"])))
    display(HTML(render_markdown(markdown, highlight=HIGHLIGHT_TELLS)))
    display(HTML("<hr>"))